# 🚀 AI 驅動量化交易系統 - Phase 1: 環境搭建

## 目標
- 安裝並啟動 Ollama
- 下載輕量級 AI 模型 (Gemma3 4B)
- 建立 SQLite 資料庫架構
- 測試系統基礎功能

## 硬體需求
- GPU: T4 (Colab 免費版)
- RAM: 12GB+
- 磁碟: 20GB+

In [ ]:
# @title 🔧 步驟 1: 安裝 Ollama

print("="*60)
print("🚀 開始安裝 Ollama...")
print("="*60)

# 下載並安裝 Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n✅ Ollama 安裝完成！")

In [ ]:
# @title 🎯 步驟 2: 啟動 Ollama 服務

import subprocess
import time
import requests

print("="*60)
print("🔥 啟動 Ollama 服務...")
print("="*60)

# 後台啟動 Ollama
proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ 等待服務啟動...")
time.sleep(8)

# 檢查服務是否就緒
for i in range(10):
    try:
        response = requests.get('http://localhost:11434', timeout=2)
        print("\n✅ Ollama 服務已就緒！")
        print(f"   服務地址: http://localhost:11434")
        break
    except:
        time.sleep(2)
        print(f"   嘗試 {i+1}/10...")
else:
    print("\n❌ 服務啟動失敗，請檢查錯誤訊息")

In [ ]:
# @title 📥 步驟 3: 下載 AI 模型

MODEL_NAME = "gemma3:4b"  # @param {type:"string"}

print("="*60)
print(f"📦 下載模型: {MODEL_NAME}")
print("="*60)
print("⏳ 預計需要 3-5 分鐘，請耐心等待...\n")

!ollama pull {MODEL_NAME}

print(f"\n✅ 模型 {MODEL_NAME} 下載完成！")

# 驗證模型
print("\n🧪 驗證模型...")
!ollama list

In [ ]:
# @title 🧪 步驟 4: 測試 AI 模型

import json

def test_ollama(prompt, model='gemma3:4b'):
    """測試 Ollama 模型"""
    url = 'http://localhost:11434/api/generate'
    
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': 0.3,
            'num_ctx': 2048
        }
    }
    
    response = requests.post(url, json=payload, timeout=60)
    result = response.json()
    
    return result['response'], result.get('eval_count', 0)

# 測試提示詞
test_prompt = """你是一個股票分析師。SPY 當前價格 $450，RSI 為 65。
請用一句話（不超過20字）評估當前市場狀態。
"""

print("="*60)
print("🧪 測試 AI 決策能力")
print("="*60)
print(f"\n提示詞:\n{test_prompt}")
print("\n⏳ 正在生成回應...\n")

response, tokens = test_ollama(test_prompt)

print("="*60)
print("🤖 AI 回應:")
print("="*60)
print(response)
print(f"\n📊 使用 Token 數: {tokens}")
print("\n✅ AI 模型運作正常！")

In [ ]:
# @title 🗄️ 步驟 5: 建立 SQLite 資料庫

import sqlite3
from datetime import datetime

# 資料庫路徑
DB_PATH = '/content/ai_trading_system.db'

print("="*60)
print("🗄️ 建立資料庫架構")
print("="*60)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# 表 1: 每日市場數據
cursor.execute("""
CREATE TABLE IF NOT EXISTS daily_data (
    date DATE NOT NULL,
    symbol VARCHAR(10) NOT NULL,
    open REAL,
    high REAL,
    low REAL,
    close REAL,
    volume INTEGER,
    
    -- 技術指標
    sma_20 REAL,
    sma_50 REAL,
    sma_200 REAL,
    rsi_14 REAL,
    macd REAL,
    macd_signal REAL,
    
    -- 市場環境
    spy_return REAL,
    vix REAL,
    market_regime TEXT,
    
    PRIMARY KEY (date, symbol)
)
""")
print("✅ 表 1: daily_data 已建立")

# 表 2: AI 分析結果
cursor.execute("""
CREATE TABLE IF NOT EXISTS ai_analysis (
    date DATE NOT NULL,
    symbol VARCHAR(10) NOT NULL,
    
    -- AI 分析
    market_view TEXT,
    technical_score REAL,
    risk_level TEXT,
    
    -- 決策
    action TEXT,
    position_size REAL,
    confidence REAL,
    reasoning TEXT,
    
    -- 元數據
    model_used TEXT,
    tokens_used INTEGER,
    processing_time REAL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    
    PRIMARY KEY (date, symbol)
)
""")
print("✅ 表 2: ai_analysis 已建立")

# 表 3: 回測交易記錄
cursor.execute("""
CREATE TABLE IF NOT EXISTS backtest_trades (
    trade_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date DATE NOT NULL,
    symbol VARCHAR(10) NOT NULL,
    action TEXT,
    shares REAL,
    price REAL,
    position_value REAL,
    portfolio_value REAL,
    cash REAL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")
print("✅ 表 3: backtest_trades 已建立")

# 表 4: 績效指標
cursor.execute("""
CREATE TABLE IF NOT EXISTS performance_metrics (
    date DATE PRIMARY KEY,
    total_value REAL,
    daily_return REAL,
    cumulative_return REAL,
    max_drawdown REAL,
    sharpe_ratio REAL,
    win_rate REAL
)
""")
print("✅ 表 4: performance_metrics 已建立")

conn.commit()

# 驗證表結構
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()

print("\n" + "="*60)
print("📊 資料庫結構概覽")
print("="*60)
print(f"資料庫位置: {DB_PATH}")
print(f"\n已建立的表格:")
for i, table in enumerate(tables, 1):
    print(f"  {i}. {table[0]}")

conn.close()

print("\n✅ 資料庫架構建立完成！")

# 儲存資料庫路徑供後續 Notebook 使用
with open('/content/config.txt', 'w') as f:
    f.write(f"DB_PATH={DB_PATH}\n")
    f.write(f"MODEL_NAME={MODEL_NAME}\n")
    f.write(f"CREATED_AT={datetime.now().isoformat()}\n")

print("\n💾 配置已保存到 /content/config.txt")

In [ ]:
# @title 📦 步驟 6: 安裝必要的 Python 套件

print("="*60)
print("📦 安裝 Python 套件")
print("="*60)

!pip install -q yfinance pandas numpy tqdm

print("\n✅ 套件安裝完成！")

# 驗證安裝
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm

print("\n📚 已導入套件:")
print(f"  - yfinance: {yf.__version__}")
print(f"  - pandas: {pd.__version__}")
print(f"  - numpy: {np.__version__}")

In [ ]:
# @title 🎉 步驟 7: 系統狀態總結

import subprocess

print("="*60)
print("🎉 環境搭建完成！系統狀態總結")
print("="*60)

print("\n✅ Ollama 服務")
try:
    response = requests.get('http://localhost:11434', timeout=2)
    print("   狀態: 運行中 ✓")
except:
    print("   狀態: 未運行 ✗")

print("\n✅ AI 模型")
result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print("   已下載模型:")
for line in result.stdout.split('\n')[1:]:
    if line.strip():
        print(f"   - {line.split()[0]}")

print("\n✅ 資料庫")
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print(f"   位置: {DB_PATH}")
print(f"   表格數: {len(tables)}")
conn.close()

print("\n✅ Python 套件")
print("   - yfinance ✓")
print("   - pandas ✓")
print("   - numpy ✓")
print("   - requests ✓")

print("\n" + "="*60)
print("📝 下一步")
print("="*60)
print("1. 執行 Notebook 2: 數據收集與處理")
print("2. 下載 ETF 歷史數據並計算技術指標")
print("3. 開始 AI 決策測試")
print("\n🚀 準備就緒！可以開始下一階段了。")

---

## 📋 檢查清單

- [ ] Ollama 已安裝並啟動
- [ ] Gemma3 4B 模型已下載
- [ ] AI 模型測試通過
- [ ] SQLite 資料庫已建立
- [ ] Python 套件已安裝

## 🔧 故障排除

**問題 1: Ollama 服務無法啟動**
- 重新執行「步驟 2」
- 檢查 Colab GPU 是否啟用

**問題 2: 模型下載失敗**
- 檢查網路連線
- 嘗試下載更小的模型（如 `gemma3:1b`）

**問題 3: 記憶體不足**
- 使用更小的模型
- 重新啟動 Colab Runtime

## 📚 資源

- [Ollama 官方文檔](https://ollama.com/)
- [Gemma 模型資訊](https://ollama.com/library/gemma3)
- [yfinance 文檔](https://pypi.org/project/yfinance/)